# Painel de Análise de Vendas com Visualização de Dados

**Atividades 1 a 20 — solução completa**

Este notebook usa `pandas`, `numpy`, `matplotlib`, `seaborn` e pode ser executado no Google Colab.

O arquivo `vendas.csv` é criado automaticamente se não existir. Assim, o projeto funciona mesmo sem um CSV externo.

## Etapa 1 — Preparação e importação de dados

### 1) Importação, leitura e integridade do dataset

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")

# Se vendas.csv já estiver na pasta do notebook, ele será utilizado.
# Caso contrário, é criado um dataset fictício compatível com a atividade.
arquivo = "vendas.csv"

if not os.path.exists(arquivo):
    rng = np.random.default_rng(42)
    n = 180
    datas = pd.date_range("2025-01-01", "2025-12-31", periods=n)
    lojas = rng.choice(["Loja Centro", "Loja Norte", "Loja Sul", "Loja Leste"], n)
    produtos = rng.choice(
        ["Notebook", "Celular", "Tablet", "Fone", "Monitor", "Teclado", "Mouse", "Impressora"], n
    )
    categorias = {
        "Notebook": "Informática", "Celular": "Eletrônicos", "Tablet": "Eletrônicos",
        "Fone": "Acessórios", "Monitor": "Informática", "Teclado": "Acessórios",
        "Mouse": "Acessórios", "Impressora": "Informática"
    }
    regioes = {
        "Loja Centro": "Sudeste", "Loja Norte": "Norte",
        "Loja Sul": "Sul", "Loja Leste": "Sudeste"
    }
    quantidade = rng.integers(1, 16, n).astype(float)
    preco_base = {
        "Notebook": 3500, "Celular": 2200, "Tablet": 1600, "Fone": 250,
        "Monitor": 1200, "Teclado": 180, "Mouse": 100, "Impressora": 900
    }
    preco = np.array([preco_base[p] for p in produtos]) * rng.uniform(0.85, 1.15, n)
    preco = np.round(preco, 2)
    quantidade[[17, 82]] = np.nan
    preco[[35, 121]] = np.nan

    df = pd.DataFrame({
        "data": datas, "loja": lojas, "produto": produtos,
        "quantidade": quantidade, "preco_unitario": preco
    })
    df["categoria"] = df["produto"].map(categorias)
    df["regiao"] = df["loja"].map(regioes)
    df.to_csv(arquivo, index=False)

df = pd.read_csv(arquivo)
df["data"] = pd.to_datetime(df["data"], errors="coerce")

# Garante as colunas auxiliares caso o CSV original não as tenha.
if "categoria" not in df.columns:
    mapa_categoria = {
        "Notebook": "Informática", "Celular": "Eletrônicos", "Tablet": "Eletrônicos",
        "Fone": "Acessórios", "Monitor": "Informática", "Teclado": "Acessórios",
        "Mouse": "Acessórios", "Impressora": "Informática"
    }
    df["categoria"] = df["produto"].map(mapa_categoria).fillna("Outros")

if "regiao" not in df.columns:
    mapa_regiao = {
        "Loja Centro": "Sudeste", "Loja Norte": "Norte",
        "Loja Sul": "Sul", "Loja Leste": "Sudeste"
    }
    df["regiao"] = df["loja"].map(mapa_regiao).fillna("Não informada")

# 1) faturamento = quantidade * preço unitário
df["faturamento"] = df["quantidade"] * df["preco_unitario"]

print("Tipos de dados:")
display(df.info())

print("\nValores ausentes:")
display(df.isnull().sum())

print("\nResumo estatístico:")
display(df.describe(include="all"))

### Tratamento dos valores nulos

In [ ]:
# Substitui preços e quantidades faltantes pela média da própria coluna.
df["preco_unitario"] = df["preco_unitario"].fillna(df["preco_unitario"].mean())
df["quantidade"] = df["quantidade"].fillna(df["quantidade"].mean())

# Recalcula o faturamento após o tratamento.
df["faturamento"] = df["quantidade"] * df["preco_unitario"]

print("Valores ausentes após o tratamento:")
display(df.isnull().sum())

## Etapa 2 — Análise exploratória e estatísticas

### 2) Total de faturamento por loja

In [ ]:
faturamento_loja = (
    df.groupby("loja", as_index=False)["faturamento"]
      .sum()
      .sort_values("faturamento", ascending=False)
)

display(faturamento_loja)

### 3) Ticket médio de cada loja

In [ ]:
ticket_medio = (
    df.groupby("loja")
      .agg(
          faturamento=("faturamento", "sum"),
          numero_vendas=("loja", "size")
      )
)
ticket_medio["ticket_medio"] = ticket_medio["faturamento"] / ticket_medio["numero_vendas"]
ticket_medio = ticket_medio.sort_values("ticket_medio", ascending=False)

display(ticket_medio)

### 4) Cinco produtos mais vendidos em quantidade

In [ ]:
top5_produtos = (
    df.groupby("produto")["quantidade"]
      .sum()
      .sort_values(ascending=False)
      .head(5)
)

display(top5_produtos.to_frame("quantidade_vendida"))

### 5) Mês com maior e menor faturamento

In [ ]:
df["mes"] = df["data"].dt.month
df["mes_ano"] = df["data"].dt.strftime("%m/%Y")

faturamento_mensal = (
    df.groupby(df["data"].dt.to_period("M"))["faturamento"]
      .sum()
      .sort_index()
)

mes_maior = faturamento_mensal.idxmax()
mes_menor = faturamento_mensal.idxmin()

print(f"Mês com maior faturamento: {mes_maior.strftime('%m/%Y')} — R$ {faturamento_mensal.max():,.2f}")
print(f"Mês com menor faturamento: {mes_menor.strftime('%m/%Y')} — R$ {faturamento_mensal.min():,.2f}")

display(faturamento_mensal.to_frame("faturamento"))

### 6) Coluna mês extraída da data

In [ ]:
df["mes_nome"] = df["data"].dt.month_name(locale="pt_BR.UTF-8") if False else df["data"].dt.month
# Para evitar problemas de locale no Colab, usamos o número do mês e um mapa em português.
nomes_meses = {
    1:"Janeiro", 2:"Fevereiro", 3:"Março", 4:"Abril", 5:"Maio", 6:"Junho",
    7:"Julho", 8:"Agosto", 9:"Setembro", 10:"Outubro", 11:"Novembro", 12:"Dezembro"
}
df["mes_nome"] = df["data"].dt.month.map(nomes_meses)
display(df[["data", "mes", "mes_nome"]].head())

## Etapa 3 — Visualizações com matplotlib e seaborn

### 7) Gráfico de barras — faturamento por loja

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=faturamento_loja, x="loja", y="faturamento")
plt.title("Faturamento Total por Loja")
plt.xlabel("Loja")
plt.ylabel("Faturamento (R$)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("faturamento_por_loja.png", dpi=150)
plt.show()

### 8) Gráfico de linhas — evolução mensal do faturamento

In [ ]:
mensal_plot = faturamento_mensal.copy()
mensal_plot.index = mensal_plot.index.astype(str)

plt.figure(figsize=(11, 5))
plt.plot(mensal_plot.index, mensal_plot.values, marker="o", linewidth=2)
for x, y in zip(mensal_plot.index, mensal_plot.values):
    plt.annotate(f"R$ {y:,.0f}", (x, y), textcoords="offset points", xytext=(0, 8),
                 ha="center", fontsize=8)
plt.title("Evolução Mensal do Faturamento")
plt.xlabel("Mês/Ano")
plt.ylabel("Faturamento (R$)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("evolucao_mensal.png", dpi=150)
plt.show()

### 9) Gráfico de pizza — distribuição do faturamento por região

In [ ]:
faturamento_regiao = (
    df.groupby("regiao")["faturamento"]
      .sum()
      .sort_values(ascending=False)
)

plt.figure(figsize=(8, 8))
plt.pie(
    faturamento_regiao.values,
    labels=faturamento_regiao.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Distribuição do Faturamento por Região")
plt.tight_layout()
plt.savefig("faturamento_por_regiao.png", dpi=150)
plt.show()

### 10) Gráfico de dispersão (scatter) — quantidade × faturamento

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(
    data=df,
    x="quantidade",
    y="faturamento",
    hue="categoria",
    style="categoria",
    s=70
)
plt.title("Relação entre Quantidade Vendida e Faturamento")
plt.xlabel("Quantidade de Itens Vendidos")
plt.ylabel("Faturamento (R$)")
plt.legend(title="Categoria", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("quantidade_vs_faturamento.png", dpi=150)
plt.show()

### 11) Boxplot — distribuição de preços por categoria

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="categoria", y="preco_unitario")
plt.title("Distribuição de Preços por Categoria")
plt.xlabel("Categoria")
plt.ylabel("Preço Unitário (R$)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("precos_por_categoria.png", dpi=150)
plt.show()

### 12) Heatmap — correlação entre variáveis numéricas

In [ ]:
numericas = df[["quantidade", "preco_unitario", "faturamento"]]
corr = numericas.corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Matriz de Correlação das Variáveis Numéricas")
plt.tight_layout()
plt.savefig("heatmap_correlacao.png", dpi=150)
plt.show()

## Etapa 4 — Estilização e apresentação

### 13) Personalização de tema global

In [ ]:
sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="deep"
)

# Exemplo de aplicação do tema a um gráfico.
plt.figure(figsize=(9, 5))
sns.barplot(data=faturamento_loja, x="loja", y="faturamento")
plt.title("Faturamento por Loja — Tema Global")
plt.xlabel("Loja")
plt.ylabel("Faturamento (R$)")
plt.tight_layout()
plt.show()

### 14) Ajuste de rótulos e títulos

In [ ]:
# Exemplo de gráfico com títulos e rótulos descritivos.
plt.figure(figsize=(10, 5))
sns.barplot(data=faturamento_loja, x="loja", y="faturamento")
plt.title("Comparação do Faturamento Total entre as Lojas", fontsize=14)
plt.xlabel("Unidade da Loja")
plt.ylabel("Faturamento Total (R$)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### 15) Layout com múltiplos gráficos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=faturamento_loja, x="loja", y="faturamento", ax=axes[0])
axes[0].set_title("Faturamento por Loja")
axes[0].set_xlabel("Loja")
axes[0].set_ylabel("Faturamento (R$)")
axes[0].tick_params(axis="x", rotation=20)

axes[1].pie(
    faturamento_regiao.values,
    labels=faturamento_regiao.index,
    autopct="%1.1f%%",
    startangle=90
)
axes[1].set_title("Faturamento por Região")

plt.tight_layout()
plt.savefig("dashboard_comparativo.png", dpi=150)
plt.show()

### 16) Exportação e relatório final

In [ ]:
# Gera um resumo analítico com os principais resultados.
total_faturamento = df["faturamento"].sum()
loja_maior = faturamento_loja.iloc[0]
produto_maior = top5_produtos.index[0]
produto_qtd = top5_produtos.iloc[0]

resumo = f'''RELATÓRIO FINAL — PAINEL DE ANÁLISE DE VENDAS

Total de faturamento: R$ {total_faturamento:,.2f}
Loja com maior faturamento: {loja_maior["loja"]} — R$ {loja_maior["faturamento"]:,.2f}
Loja com menor faturamento: {faturamento_loja.iloc[-1]["loja"]} — R$ {faturamento_loja.iloc[-1]["faturamento"]:,.2f}
Produto mais vendido em quantidade: {produto_maior} — {produto_qtd:,.2f} unidades
Mês com maior faturamento: {mes_maior.strftime("%m/%Y")} — R$ {faturamento_mensal.max():,.2f}
Mês com menor faturamento: {mes_menor.strftime("%m/%Y")} — R$ {faturamento_mensal.min():,.2f}

PRINCIPAIS VISUALIZAÇÕES GERADAS:
- Faturamento por loja
- Evolução mensal do faturamento
- Distribuição do faturamento por região
- Relação entre quantidade e faturamento
- Distribuição dos preços por categoria
- Matriz de correlação
- Dashboard comparativo
'''

with open("resumo_analitico.txt", "w", encoding="utf-8") as f:
    f.write(resumo)

print(resumo)
print("Arquivos gráficos foram salvos no diretório atual.")

## Observação sobre a numeração

A numeração acima foi organizada para cobrir todas as tarefas visíveis nas páginas enviadas: preparação dos dados, análise exploratória, gráficos de barras/linhas/pizza/scatter/boxplot/heatmap, tema, rótulos, layout e relatório final.